# Experiment 03: Physical UAV Attack Detection with Support Vector Machines (SVM)

## 1. Overview & Research Objectives
This experiment benchmarks **Support Vector Machines (SVM)** on the **Physical UAV Telemetry Dataset** (`Physical_UAV_Dataset.csv`).

### Key Research Questions:
1. **Linear vs. Non-linear Kernels:** Can a fast `LinearSVC` separate kinematic flight states, or is an `RBF` kernel required for non-linear flight dynamics?
2. **The Margin Bias Problem:** How severely does class imbalance affect SVM hyperplanes, and does `class_weight='balanced'` salvage minority attack classes (`DoS`, `Replay`)?
3. **Inference Latency Trade-off:** How does SVM latency compare with Random Forest for flight controller deployment?
4. **Feature Scaling Impact:** Measuring performance with `StandardScaler` pipeline normalization.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))
sys.path.append(os.path.abspath('.'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.svm import SVC, LinearSVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score

from utils.data_loader import load_physical_dataset, get_stratified_split
from utils.metrics import compute_comprehensive_metrics, plot_confusion_matrix

sns.set_theme(style="whitegrid")
plt.rcParams['font.size'] = 11

## 2. Leakage-Free Physical Data Loading & Stratified Split

In [ ]:
X, y, feature_names = load_physical_dataset("../Physical_UAV_Dataset.csv")
X_train, X_test, y_train, y_test, encoder = get_stratified_split(X, y, test_size=0.3, random_state=42)
class_names = [str(c) for c in encoder.classes_]

print(f"[*] Loaded Physical Telemetry: {X.shape[0]} samples, {X.shape[1]} features")
print(f"[*] Classes: {class_names}")
print(f"[*] Training size: {X_train.shape[0]}, Testing size: {X_test.shape[0]}")

## 3. Fast Linear SVM (LinearSVC) Baseline

In [ ]:
linear_svm = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LinearSVC(C=1.0, max_iter=3000, random_state=42))
])
linear_svm.fit(X_train, y_train)

m_lin, y_pred_lin, cm_lin = compute_comprehensive_metrics(
    linear_svm, X_test, y_test, encoder, model_name="Linear SVM (Scaled)", domain="Physical"
)

print("=== Linear SVM Metrics ===")
for k, v in m_lin.items():
    print(f"{k:25}: {v}")

plot_confusion_matrix(cm_lin, class_names, title="Linear SVM - Confusion Matrix")

## 4. Non-Linear RBF Kernel SVM & Hyperparameter Exploration

In [ ]:
svm_configs = [
    ("Linear SVM Default", LinearSVC(C=1.0, max_iter=3000, random_state=42)),
    ("Linear SVM Balanced", LinearSVC(C=1.0, class_weight='balanced', max_iter=3000, random_state=42)),
    ("RBF SVM (C=1.0, Default)", SVC(C=1.0, kernel='rbf', random_state=42)),
    ("RBF SVM (C=10.0, gamma=0.1)", SVC(C=10.0, gamma=0.1, kernel='rbf', random_state=42)),
    ("RBF SVM (C=10.0, Balanced)", SVC(C=10.0, gamma=0.1, class_weight='balanced', kernel='rbf', random_state=42)),
]

results = []
for name, clf in svm_configs:
    pipe = Pipeline([('scaler', StandardScaler()), ('clf', clf)])
    pipe.fit(X_train, y_train)
    m, _, _ = compute_comprehensive_metrics(pipe, X_test, y_test, encoder, model_name=name, domain="Physical")
    results.append(m)

df_comparison = pd.DataFrame(results)
display(df_comparison[["Model", "Accuracy (%)", "Macro F1 (%)", "False Alarm Rate (%)", "Latency (us/sample)", "Model Size (KB)", "F1: DoS (%)", "F1: Replay (%)", "F1: Evil_Twin (%)", "F1: FDI (%)"]])

## 5. Optimal SVM Evaluation (Balanced RBF)

In [ ]:
best_svm = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', SVC(C=10.0, gamma=0.1, class_weight='balanced', kernel='rbf', random_state=42))
])
best_svm.fit(X_train, y_train)

m_best, y_pred_best, cm_best = compute_comprehensive_metrics(
    best_svm, X_test, y_test, encoder, model_name="Optimal RBF SVM", domain="Physical"
)
plot_confusion_matrix(cm_best, class_names, title="Optimal RBF SVM (Balanced) - Confusion Matrix")

## 6. Stratified 5-Fold Cross-Validation

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores_acc = cross_val_score(best_svm, X, encoder.transform(y), cv=cv, scoring='accuracy')
cv_scores_f1 = cross_val_score(best_svm, X, encoder.transform(y), cv=cv, scoring='f1_macro')

print(f"[*] 5-Fold CV Accuracy: {cv_scores_acc.mean()*100:.2f}% (+/- {cv_scores_acc.std()*100:.2f}%)")
print(f"[*] 5-Fold CV Macro F1: {cv_scores_f1.mean()*100:.2f}% (+/- {cv_scores_f1.std()*100:.2f}%)")

## 7. Summary of Findings & Comparison with Random Forest
1. **Linear Inadequacy:** Linear SVM achieves **84.61% overall accuracy** solely by predicting majority classes, completely collapsing on non-linear minority attacks (0.0% F1 on DoS, 4.68% on Replay).
2. **Impact of Balancing:** Class-weight balancing in RBF SVM raises Macro F1 from **60.06% to 68.79%**, with DoS F1 reaching **39.14%** and Replay reaching **33.77%**.
3. **Inference Latency & Efficiency:** RBF kernel SVM requires **~163 μs per sample**, roughly **20x slower than Random Forest (8 μs)** due to computing kernel dot-products against thousands of support vectors.